## Auto-Grading Cells
### Append these cells after the student's notebook and run all cells.

- Each question produces a `qN_check.csv`.
- The summary cell at the bottom prints pass rates.
- All test inputs are seeded so results are deterministic.

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
import random as _random

### Q1 Check — Average Angle between Random Vectors

In [ ]:
def _ref_calculate_expected_angle(D, num_samples=1000):
    v1 = np.random.rand(num_samples, D)
    v2 = np.random.rand(num_samples, D)
    dot_products = np.sum(v1 * v2, axis=1)
    norms_v1 = np.linalg.norm(v1, axis=1)
    norms_v2 = np.linalg.norm(v2, axis=1)
    cos_angles = dot_products / (norms_v1 * norms_v2)
    cos_angles = np.clip(cos_angles, -1, 1)
    angles = np.arccos(cos_angles)
    return np.mean(np.degrees(angles))

test_dims_q1 = np.linspace(2, 1000, 100, dtype=int)
results_q1 = []

for D in test_dims_q1:
    seed = 1000 + int(D)
    np.random.seed(seed)
    expected = _ref_calculate_expected_angle(D)
    np.random.seed(seed)
    try:
        actual = calculate_expected_angle(D)
        correct = abs(expected - actual) < 1e-4
    except Exception as e:
        actual = str(e)
        correct = False
    results_q1.append({
        'dimension': int(D),
        'expected_angle': round(expected, 6) if isinstance(expected, (int, float)) else expected,
        'actual_angle': round(actual, 6) if isinstance(actual, (int, float, np.floating)) else actual,
        'correct': correct
    })

pd.DataFrame(results_q1).to_csv('q1_check.csv', index=False)
_p = sum(r['correct'] for r in results_q1)
print(f"Q1: {_p}/100 passed")

### Q2 Check — kNN Majority Vote

In [ ]:
def _ref_get_majority_vote(labels):
    return Counter(labels).most_common(1)[0][0]

# 100 test cases — odd k with binary labels ⇒ no ties
_random.seed(42)
_test_cases_q2 = []
for _ in range(100):
    k = _random.choice([3, 5, 7, 9, 11])
    labels = [_random.choice(["Comedy", "Action"]) for _ in range(k)]
    _test_cases_q2.append(labels)

results_q2 = []
for i, labels in enumerate(_test_cases_q2):
    expected = _ref_get_majority_vote(labels)
    try:
        actual = get_majority_vote(list(labels))
        correct = (actual == expected)
    except Exception as e:
        actual = str(e)
        correct = False
    results_q2.append({
        'test_id': i,
        'input': str(labels),
        'expected': expected,
        'actual': actual,
        'correct': correct
    })

pd.DataFrame(results_q2).to_csv('q2_check.csv', index=False)
_p = sum(r['correct'] for r in results_q2)
print(f"Q2: {_p}/100 passed")

### Q3 Check — Gaussian Naive Bayes

In [ ]:
def _ref_gaussian_nb_predict(X_train, y_train, x_query):
    classes = np.unique(y_train)
    best_class, best_log_prob = None, -np.inf
    for c in classes:
        X_c = X_train[y_train == c]
        log_prior = np.log(len(X_c) / len(y_train))
        mean_c = X_c.mean(axis=0)
        var_c  = X_c.var(axis=0)
        log_likelihood = np.sum(
            -0.5 * np.log(2 * np.pi * var_c)
            - 0.5 * (x_query - mean_c)**2 / var_c
        )
        total = log_prior + log_likelihood
        if total > best_log_prob:
            best_log_prob = total
            best_class = c
    return best_class

# Fixed dataset: two 3-D Gaussian blobs
np.random.seed(42)
_npc = 50
_X0 = np.random.randn(_npc, 3) + np.array([0, 0, 0])
_X1 = np.random.randn(_npc, 3) + np.array([3, 3, 3])
_Xtr3 = np.vstack([_X0, _X1])
_ytr3 = np.array([0]*_npc + [1]*_npc)

# 100 query points spread around the decision boundary
np.random.seed(123)
_Xq3 = np.random.randn(100, 3) * 2 + 1.5

results_q3 = []
for i in range(100):
    xq = _Xq3[i]
    expected = _ref_gaussian_nb_predict(_Xtr3, _ytr3, xq)
    try:
        actual = gaussian_nb_predict(_Xtr3, _ytr3, xq)
        correct = (actual == expected)
    except Exception as e:
        actual = str(e)
        correct = False
    results_q3.append({
        'test_id': i,
        'query': str(np.round(xq, 4).tolist()),
        'expected': int(expected) if not isinstance(expected, str) else expected,
        'actual': int(actual) if not isinstance(actual, str) else actual,
        'correct': correct
    })

pd.DataFrame(results_q3).to_csv('q3_check.csv', index=False)
_p = sum(r['correct'] for r in results_q3)
print(f"Q3: {_p}/100 passed")

### Q4 Check — Logistic Regression
*(sigmoid is restored to the correct Q4 version before grading,
since Q5 may have overwritten it with the student's fill-in-the-blank version)*

In [ ]:
# Save student's Q5 sigmoid, restore correct Q4 version for this cell
try:
    _saved_sigmoid = sigmoid
except NameError:
    _saved_sigmoid = None

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def _ref_train_logistic(X, y, lr=0.1, epochs=1000):
    n, d = X.shape
    w = np.zeros((d, 1))
    for _ in range(epochs):
        z = X.dot(w)
        a = sigmoid(z)
        dw = (1/n) * X.T.dot(a - y)
        w = w - lr * dw
    return w

# Fixed linearly separable dataset
np.random.seed(42)
_n4 = 200
_Xp = np.random.randn(_n4 // 2, 2) + np.array([2, 2])
_Xn = np.random.randn(_n4 // 2, 2) + np.array([-2, -2])
_Xtr4 = np.vstack([_Xp, _Xn])
_ytr4 = np.vstack([np.ones((_n4 // 2, 1)), np.zeros((_n4 // 2, 1))])

_w_ref = _ref_train_logistic(_Xtr4, _ytr4)

try:
    _w_stu = train_logistic(_Xtr4, _ytr4)
    _q4_trained = True
except Exception as e:
    _w_stu = None
    _q4_trained = False
    print(f"Q4 train_logistic error: {e}")

# 100 test points
np.random.seed(123)
_Xt4 = np.random.randn(100, 2) * 3

results_q4 = []
for i in range(100):
    x = _Xt4[i:i+1]
    prob_ref = float(sigmoid(x.dot(_w_ref))[0, 0])
    pred_ref = int(prob_ref >= 0.5)
    try:
        prob_stu = float(sigmoid(x.dot(_w_stu))[0, 0])
        pred_stu = int(prob_stu >= threshold)
        correct = (pred_ref == pred_stu) and abs(prob_ref - prob_stu) < 0.05
    except Exception as e:
        prob_stu = str(e)
        pred_stu = str(e)
        correct = False
    results_q4.append({
        'test_id': i,
        'expected_prob': round(prob_ref, 6),
        'actual_prob': round(prob_stu, 6) if isinstance(prob_stu, float) else prob_stu,
        'expected_class': pred_ref,
        'actual_class': pred_stu,
        'correct': correct
    })

pd.DataFrame(results_q4).to_csv('q4_check.csv', index=False)
_p = sum(r['correct'] for r in results_q4)
print(f"Q4: {_p}/100 passed")

# Restore student's Q5 sigmoid for Q5 grading
if _saved_sigmoid is not None:
    sigmoid = _saved_sigmoid

### Q5 Check — Neural Network (Forward + Backprop)

In [ ]:
# --- Reference implementations (prefixed to avoid collision) ---
def _ref_sig(z):
    return 1 / (1 + np.exp(-z))

def _ref_sig_d(a):
    return a * (1 - a)

def _ref_fwd(X, W1, b1, W2, b2, W3, b3):
    z1 = X.dot(W1) + b1;  a1 = _ref_sig(z1)
    z2 = a1.dot(W2) + b2; a2 = _ref_sig(z2)
    z3 = a2.dot(W3) + b3; y_hat = z3
    return z1, a1, z2, a2, z3, y_hat

def _ref_sse(y, yh):
    return np.sum((y - yh) ** 2)

def _ref_trn(X, y, W1, b1, W2, b2, W3, b3, lr=0.01, epochs=100):
    for _ in range(epochs):
        z1, a1, z2, a2, z3, yh = _ref_fwd(X, W1, b1, W2, b2, W3, b3)
        dz3 = -2 * (y - yh)
        dW3 = a2.T.dot(dz3);  db3 = dz3.sum(axis=0, keepdims=True)
        da2 = dz3.dot(W3.T);  dz2 = da2 * _ref_sig_d(a2)
        dW2 = a1.T.dot(dz2);  db2 = dz2.sum(axis=0, keepdims=True)
        da1 = dz2.dot(W2.T);  dz1 = da1 * _ref_sig_d(a1)
        dW1 = X.T.dot(dz1);   db1 = dz1.sum(axis=0, keepdims=True)
        W3 -= lr*dW3; b3 -= lr*db3
        W2 -= lr*dW2; b2 -= lr*db2
        W1 -= lr*dW1; b1 -= lr*db1
    return W1, b1, W2, b2, W3, b3

# --- Fixed data & weights ---
np.random.seed(42)
_X5 = np.random.randn(100, 5)
_y5 = np.random.randn(100, 1)
_W1i = np.random.randn(5, 2) * 0.1
_b1i = np.zeros((1, 2))
_W2i = np.random.randn(2, 3) * 0.1
_b2i = np.zeros((1, 3))
_W3i = np.random.randn(3, 1) * 0.1
_b3i = np.zeros((1, 1))

# --- Reference forward ---
_, _, _, _, _, _yh_ref = _ref_fwd(
    _X5, _W1i.copy(), _b1i.copy(), _W2i.copy(), _b2i.copy(), _W3i.copy(), _b3i.copy())

# --- Student forward ---
try:
    _, _, _, _, _, _yh_stu = forward(
        _X5, _W1i.copy(), _b1i.copy(), _W2i.copy(), _b2i.copy(), _W3i.copy(), _b3i.copy())
    _fwd_ok = True
except Exception as e:
    _yh_stu = np.full((100, 1), np.nan)
    _fwd_ok = False
    print(f"Q5 forward error: {e}")

# --- Reference training ---
_W1r, _b1r, _W2r, _b2r, _W3r, _b3r = _ref_trn(
    _X5, _y5,
    _W1i.copy(), _b1i.copy(), _W2i.copy(), _b2i.copy(), _W3i.copy(), _b3i.copy(),
    lr=0.001, epochs=200)
_, _, _, _, _, _yh_ref_t = _ref_fwd(_X5, _W1r, _b1r, _W2r, _b2r, _W3r, _b3r)

# --- Student training ---
try:
    _W1s, _b1s, _W2s, _b2s, _W3s, _b3s = train(
        _X5, _y5,
        _W1i.copy(), _b1i.copy(), _W2i.copy(), _b2i.copy(), _W3i.copy(), _b3i.copy(),
        lr=0.001, epochs=200)
    _, _, _, _, _, _yh_stu_t = forward(
        _X5, _W1s, _b1s, _W2s, _b2s, _W3s, _b3s)
    _trn_ok = True
except Exception as e:
    _yh_stu_t = np.full((100, 1), np.nan)
    _trn_ok = False
    print(f"Q5 train error: {e}")

# --- Build CSV ---
results_q5 = []
for i in range(100):
    ef = round(float(_yh_ref[i, 0]), 6)
    af = round(float(_yh_stu[i, 0]), 6) if _fwd_ok else "error"
    fc = (abs(ef - float(af)) < 1e-4) if _fwd_ok else False

    et = round(float(_yh_ref_t[i, 0]), 6)
    at = round(float(_yh_stu_t[i, 0]), 6) if _trn_ok else "error"
    tc = (abs(et - float(at)) < 0.01) if _trn_ok else False

    results_q5.append({
        'test_id': i,
        'expected_fwd': ef, 'actual_fwd': af, 'fwd_correct': fc,
        'expected_trained': et, 'actual_trained': at, 'train_correct': tc
    })

pd.DataFrame(results_q5).to_csv('q5_check.csv', index=False)
_fp = sum(r['fwd_correct'] for r in results_q5)
_tp = sum(r['train_correct'] for r in results_q5)
print(f"Q5 Forward:  {_fp}/100 passed")
print(f"Q5 Training: {_tp}/100 passed")

### Grading Summary

In [ ]:
print("=" * 50)
print("GRADING SUMMARY")
print("=" * 50)
for q, fname in [("Q1", "q1_check.csv"), ("Q2", "q2_check.csv"),
                  ("Q3", "q3_check.csv"), ("Q4", "q4_check.csv"),
                  ("Q5", "q5_check.csv")]:
    try:
        df = pd.read_csv(fname)
        correct_cols = [c for c in df.columns if "correct" in c.lower()]
        for col in correct_cols:
            passed = int(df[col].sum())
            total = len(df)
            pct = 100 * passed / total
            print(f"  {q:>2s} ({col:<15s}): {passed:>3d}/100  ({pct:.0f}%)")
    except Exception as e:
        print(f"  {q}: ERROR - {e}")
print("=" * 50)